just to test the utils if it is visible or not

In [ ]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme.git
%cd /content/Dual_watermarking_Scheme

Cloning into 'Dual_watermarking_Scheme'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 14 (delta 1), reused 14 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), done.
Resolving deltas: 100% (1/1), done.


In [ ]:
from huggingface_hub import login

login()

In [ ]:
import pandas as pd
import numpy as np
import torch

import os
import hmac
import hashlib

from transformers import LogitsProcessor,AutoTokenizer,AutoModelForCausalLM

In [30]:
print(torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

2.11.0+cu128
cuda


In [31]:
torch.manual_seed(38)
def generate_key(size:int = 32): 
    key = os.urandom(32)
    return key

key = generate_key()
print(key)

b'\xbe\xb4@\x84\xa6\xfb\xf4\xa50\xaa\xd7\x92 _\x9fR}\xc3\xf9\xb3\x97s5D\xea \x18\x1c\x96\x12\x1cp'


In [32]:
def derive_seed(key,prev_tokens,prev_tokens_size):
    """
    prev_tokens are the list of prev tokens
    prev_tokens_size is the selected token in order to produce the seed
    returns a hash which will be used as a seed
    """
    context = list(prev_tokens[-prev_tokens_size:])

    msg = ",".join(map(str,context)).encode("utf-8")

    byte_hmacs = hmac.new(key,msg,hashlib.sha256).digest()

    hashed = int.from_bytes(byte_hmacs[:4],byteorder="big")

    return hashed

In [33]:
derive_seed(key,[432,32,124,32],2)

3445120436

In [34]:
def derive_signal_bit(key,prev_tokens,prev_tokens_size):
    """
    deterministically returns either 1 or 0
    """
    ctxt = list(prev_tokens[-prev_tokens_size:])
    msg = ",".join(map(str,ctxt)).encode("utf-8")
    d = hmac.new(key,msg,hashlib.sha256).digest()

    return d[0] & 1

In [35]:
def derive_set(
        vocab_size,
        key,
        prev_tokens,
        green_fraction=0.5,
        prev_tokens_size =1
):
    seed = derive_seed(key,prev_tokens,prev_tokens_size)

    state = np.random.RandomState(seed)

    permutation = state.permutation(vocab_size)

    green_size = int(green_fraction* vocab_size)

    return set(permutation[:green_size])

VERIFY THIS PHASE  of random generation, seed and derive_set

In [36]:
key = generate_key()
token_history = [345,21,980,32,534]

A = derive_set(
    vocab_size=5000,
    key=key,
    prev_tokens=token_history,
)

B = derive_set(
    vocab_size=5000,
    key=key,
    prev_tokens=token_history
)

print(f"A is {len(A)}")
print(A==B)

A is 2500
True


In [37]:
key = generate_key()
token_history = [345,21,980,32,534]

key2 = os.urandom(32)

A = derive_set(
    vocab_size=5000,
    key=key,
    prev_tokens=token_history,
)

B = derive_set(
    vocab_size=5000,
    key=key2,
    prev_tokens=token_history
)
print(A == B)

False


In [ ]:
#Private Logit Processor
class PrivateWatermarkProcessor(LogitsProcessor):
    def __init__(self,key,vocab_size,green_fraction=0.5,delta_private=0.7,prev_token_size=5):
        self.key = key
        self.vocab_size=vocab_size
        self.green_fraction = green_fraction
        self.delta_private = delta_private
        self.prev_token_size = prev_token_size

    def __call__(self,input_ids,scores):
        """"
        internally this receives input_ids -> tokens generated so far
        scores -> model's next-token logits
        """

        batch_size = input_ids.shape[0]

        for b in range(batch_size):

            history = input_ids[b].tolist()

            preferred_set = derive_set(vocab_size=self.vocab_size,
                                       green_fraction=self.green_fraction,
                                       key=self.key,
                                       prev_tokens_size=self.prev_token_size,
                                       prev_tokens=history
                                       )

            preferred = torch.tensor(list(preferred_set),device=scores.device)

            scores[b,preferred] += self.delta_private

        return scores
    

In [39]:
from transformers import AutoTokenizer,AutoModelForCausalLM,LogitsProcessorList

model_name="facebook/opt-2.7b"

In [40]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def load_model(model_name:str,):

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    vocabulary_size = len(tokenizer)
    print(f"model of {model_name} loaded with vocab_size {vocabulary_size}")

    return model,tokenizer,vocabulary_size

In [41]:
model,tokenizer,vocab_size = load_model(model_name)

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model of facebook/opt-2.7b loaded with vocab_size 50265


In [45]:
prompts_list = [
    "Explain why the sky is blue.",
    "write three tips to be productive",
    "What are the benefits of regular exercise?",
    "Give me three healthy breakfast ideas."
]

In [46]:
private_processor = PrivateWatermarkProcessor(key=key,
                                              vocab_size=vocab_size)
processors = LogitsProcessorList([private_processor])

In [48]:
model.eval()

results = []

for i,prompt in enumerate(prompts_list):

    inputs = tokenizer(prompt,return_tensors="pt").to(model.device)

    with torch.no_grad():
        plain_outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            temperature = 1.0,
            top_p=0.5
        )

        watermarked_outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            temperature=1.0,
            top_p=0.5,
            logits_processor=processors
        )

    plain_txt = tokenizer.decode(plain_outputs[0],skip_special_tokens=True)
    watermarked_text = tokenizer.decode(watermarked_outputs[0],skip_special_tokens=True)

    results.append({
        "id":i,
        "prompt":prompt,
        "plain_output":plain_txt,
        "wm_output":watermarked_text
    })

df = pd.DataFrame(results)

In [52]:
df

,id,prompt,plain_output,wm_output
0,0,Explain why the sky is blue.,"Explain why the sky is blue.\nIt's not, it's a...",Explain why the sky is blue.\nWhy is the sky b...
1,1,write three tips to be productive,write three tips to be productive and how to b...,write three tips to be productive\n\nWrite thr...
2,2,What are the benefits of regular exercise?,What are the benefits of regular exercise?\n\n...,What are the benefits of regular exercise?\n\n...
3,3,Give me three healthy breakfast ideas.,Give me three healthy breakfast ideas.\nI'd re...,Give me three healthy breakfast ideas. I'm a v...


In [56]:
print(f" plain output is {df.loc[2,"plain_output"]}")
print(f" watermarked output is {df.loc[2,"wm_output"]}")

 plain output is What are the benefits of regular exercise?

The benefits of regular exercise are numerous. Regular exercise can help you:

• Reduce your risk of heart disease

• Lower your risk of diabetes

• Improve your mood

• Reduce your risk of cancer

•
 watermarked output is What are the benefits of regular exercise?

Regular exercise can improve your health, lower your risk of heart disease, and help you lose weight. Regular exercise is a great way to improve your mental health, too.

Regular exercise is one of the best ways to help you live


In [ ]:
import pandas as pd
from scipy.stats import binomtest


def detect_private_watermark(
    text,
    tokenizer,
    key,
    vocab_size,
    green_fraction=0.5,
    prev_token_size=5,
    threshold=0.60,
):
    """
    Detects the private watermark by replaying the same preferred-set
    generation used during watermark embedding.
    """

    token_ids = tokenizer.encode(text, add_special_tokens=False)

    matches = 0
    total_positions = 0

    # Start after enough history exists
    for position in range(prev_token_size, len(token_ids)):

        history = token_ids[:position]

        preferred_set = derive_set(
            vocab_size=vocab_size,
            green_fraction=green_fraction,
            key=key,
            prev_tokens_size=prev_token_size,
            prev_tokens=history,
        )

        if token_ids[position] in preferred_set:
            matches += 1

        total_positions += 1

    ownership_score = matches / total_positions if total_positions else 0.0

    p_value = binomtest(
        k=matches,
        n=total_positions,
        p=green_fraction,
        alternative="greater",
    ).pvalue

    confirmed = (
        ownership_score > threshold
        and p_value < 0.05
    )

    return {
        "ownership_score": ownership_score,
        "match_count": matches,
        "num_positions": total_positions,
        "p_value": p_value,
        "confirmed": confirmed,
    }

In [61]:
df.columns

Index(['id', 'prompt', 'plain_output', 'wm_output'], dtype='object')

In [63]:
detection_results = []

for _, row in df.iterrows():

    result = detect_private_watermark(
        text=row["wm_output"],
        tokenizer=tokenizer,
        key=key,
        vocab_size=len(tokenizer),
        green_fraction=0.5,
        prev_token_size=5,
        threshold=0.60,
    )

    detection_results.append(result)

detection_df = pd.DataFrame(detection_results)

detection_df = pd.concat([df, detection_df], axis=1)

detection_df.head()


,id,prompt,plain_output,wm_output,ownership_score,match_count,num_positions,p_value,confirmed,ownership_score,match_count,num_positions,p_value,confirmed
0,0,Explain why the sky is blue.,"Explain why the sky is blue.\nIt's not, it's a...",Explain why the sky is blue.\nWhy is the sky b...,0.452830,24,53,0.794949,False,0.452830,24,53,0.794949,False
1,1,write three tips to be productive,write three tips to be productive and how to b...,write three tips to be productive\n\nWrite thr...,0.666667,34,51,0.012046,True,0.666667,34,51,0.012046,True
2,2,What are the benefits of regular exercise?,What are the benefits of regular exercise?\n\n...,What are the benefits of regular exercise?\n\n...,0.603774,32,53,0.084489,False,0.603774,32,53,0.084489,False
3,3,Give me three healthy breakfast ideas.,Give me three healthy breakfast ideas.\nI'd re...,Give me three healthy breakfast ideas. I'm a v...,0.538462,28,52,0.338904,False,0.538462,28,52,0.338904,False
